# Mock Interview Notebook
  
- prep EDA
- ver 1.01

## Import

In [2]:
from dotenv import load_dotenv
import pandas as pd
import sys
import os
import config

sys.path.insert(0, config.SRC_PATH)

import warnings
warnings.filterwarnings('ignore')

print(f"Project path: {config.PROJECT_PATH}")
print(f"Data path: {config.DATA_PATH}")
print(f"File: {config.FILE_PATH}")
print(f"Target: {config.TARGET_COLUMN}")



Project path: Q:\scripts\projects\Mock_Interview-July2026
Data path: Q:\scripts\projects\Mock_Interview-July2026\data
File: Q:\scripts\projects\Mock_Interview-July2026\data\loans_modified.csv
Target: loan_status


## Dataset Profile

In [3]:
# Data Shape info
data_link = config.FILE_PATH
df = pd.read_csv(data_link)
df.head()
print(df.sample(10))
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

      loan_id  gender married dependents     education self_employed  \
60   LP001238    Male     Yes         3+  Not Graduate           Yes   
148  LP001594    Male     Yes          0      Graduate            No   
465  LP002729    Male      No          1      Graduate            No   
394  LP002455    Male     Yes        NaN      Graduate            No   
435  LP002603  Female      No          0      Graduate            No   
358  LP002335  Female     Yes          0  Not Graduate            No   
261  LP001974     NaN      No          0      Graduate            No   
440       NaN    Male     Yes          2      Graduate            No   
378  LP002407  Female     Yes          0  Not Graduate           Yes   
70   LP001264    Male     NaN         3+  Not Graduate           Yes   

     applicant_income  coapplicant_income  loan_amount  loan_amount_term  \
60             7100.0                 0.0        125.0              60.0   
148            5708.0              5625.0        187.0 

In [4]:
# Missing/Null data impact
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query("missing_count > 0")

if missing_summary.empty:
    print("No missing values found.")
else:
    print(missing_summary)

                    missing_count  missing_pct
loan_id                        29         5.15
gender                         29         5.15
married                        19         3.37
dependents                     32         5.68
education                      22         3.91
self_employed                  34         6.04
applicant_income               26         4.62
coapplicant_income             34         6.04
loan_amount                    30         5.33
loan_amount_term               28         4.97
credit_history                 22         3.91
property_area                  21         3.73
loan_status                    28         4.97


In [5]:
counts = df[config.TARGET_COLUMN].value_counts()
pct = df[config.TARGET_COLUMN].value_counts(normalize=True) * 100

print(pd.DataFrame({'count': counts, 'percent': pct.round(1)}))

             count  percent
loan_status                
1.0            384     71.8
0.0            151     28.2


### Initial EDA Summary  
  
- Dataset Profile: 563 rows, 13 columns;   
- classification task (loan approval).   
- Target is imbalanced: 71.8% approved / 28.2% denied. Five rows with missing target values will need handling.  
  
- Missingness: Distributed across all features at ~4–6% each.  
    - loan_id is 5.15% missing, which is unusual for an ID column and suggests incomplete records.   
    - No single column dominates; this rules out simple "drop column" strategies.  
   
- Data Quality: 25 duplicate rows detected;  
    - will remove before modeling.  
    - No obvious outliers yet (income/loan amounts reasonably scaled),  
    - but 'dependents' has mixed string/numeric encoding ("3+" vs "0", "1", "2"). Candidate for boolean conversion.
  
- Potential Predictive Signals: 
    - credit_history (binary: meets guidelines or not) is industry standard strongest predictor.  
    - applicant_income and loan_amount are also industry standard expected positive correlation with approval.  
    - property_area (Urban/Semiurban/Rural) and demographics are secondary candidates.  

### Preprocessing Steps

#### Cleanup  
- Drops
    - Remove duplicate rows
    - Remove 'incomplete application' assumption rows:
        - missing 'loan_status' or 'loan_amount' or 'loan_id' column nulls  
        - missing or 0 amount in both applicant_income & coapplicant_income
- Encode/convert 'dependents' 4 category field  
    - and potential feature for binary version 'has_dependents'  
- Impute missing values  
    - To be determined later  
        - applicant_income (mode/median potential), 0 when coapplicant_income exists
        - coapplicant_income (0 when not married)    
        - loan_amount_term (medina potential)  
- Categoricals:  
    - gender,  
    - married Y/N (impute N)
    - education Y/N (impute Y if loan_status = approved) 
    - self_employed Y/N (impute N)
    - property_area (Urban, semi-urban, rural) 
    - Mode, Median or "Unknown"  
- credit_history: impute as 0 where missing  
- Validation:  
    - check for negative amounts in numericals  
    - check ranges of * _income(s) and loan_amount_term(s)    


#### Feature Engineering  
  
- total_income = applicant_income + coapplicant_income  
- debt_income_ratio = total_income / loan_amount  
    - loan_amount_term needed 

## Automated EDA Profiling & Config Draft

In [6]:
from profiling_utils import generate_eda_summary
from utils import create_output_folders

eda_summary = generate_eda_summary(df, config.TARGET_COLUMN)
print(eda_summary)

create_output_folders(config.REPORTS_PATH)
eda_summary_path = os.path.join(config.REPORTS_PATH, "eda_summary.txt")
with open(eda_summary_path, "w") as f:
    f.write(eda_summary)

EDA SUMMARY
Shape: 563 rows x 13 columns

Dtypes:
  loan_id: str
  gender: str
  married: str
  dependents: str
  education: str
  self_employed: str
  applicant_income: float64
  coapplicant_income: float64
  loan_amount: float64
  loan_amount_term: float64
  credit_history: float64
  property_area: str
  loan_status: float64

Missing values:
  self_employed: 34 (6.04%)
  coapplicant_income: 34 (6.04%)
  dependents: 32 (5.68%)
  loan_amount: 30 (5.33%)
  loan_id: 29 (5.15%)
  gender: 29 (5.15%)
  loan_amount_term: 28 (4.97%)
  loan_status: 28 (4.97%)
  applicant_income: 26 (4.62%)
  credit_history: 22 (3.91%)
  education: 22 (3.91%)
  property_area: 21 (3.73%)
  married: 19 (3.37%)

Duplicate rows: 25

Target balance (loan_status):
  1.0: 384 (68.2%)
  0.0: 151 (26.8%)
  nan: 28 (5.0%)

Numeric column ranges:
  applicant_income: min=150.0, max=81000.0, mean=5379.37
  coapplicant_income: min=0.0, max=41667.0, mean=1692.60
  loan_amount: min=9.0, max=650.0, mean=147.43
  loan_amount_ter

In [7]:
from profiling_utils import draft_data_config, write_data_config, summarize_unresolved

draft = draft_data_config(df, config.TARGET_COLUMN, config.FILE_NAME)
CONFIG_OUTPUT_MODE = "draft"  # use "canonical" only for a new file
if CONFIG_OUTPUT_MODE == "draft":
    data_config_path = os.path.join(config.SRC_PATH, "data_config_draft.py")
elif CONFIG_OUTPUT_MODE == "canonical":
    data_config_path = os.path.join(config.SRC_PATH, "data_config.py")
else:
    raise ValueError("CONFIG_OUTPUT_MODE must be 'draft' or 'canonical'")

write_data_config(
    draft,
    data_config_path,
    overwrite=(CONFIG_OUTPUT_MODE == "draft"),
)

unresolved = summarize_unresolved(draft)
print(unresolved)

with open(eda_summary_path, "a") as f:
    f.write("\n\n" + "=" * 60 + "\n")
    f.write(f"UNRESOLVED ITEMS (needs_review) — see {data_config_path.name}\n")
    f.write("=" * 60 + "\n")
    f.write(unresolved.to_string(index=False))

                   key                                             reason
0  IMPUTATION_STRATEGY  requires domain judgment on how each column's ...
1      FEATURE_COLUMNS  requires deciding which columns to retain/engi...
2         ENCODING_MAP  requires deciding encoding strategy (ordinal v...
3       NOISY_FEATURES  requires manual inspection to flag unreliable ...
4    VALIDATION_BOUNDS  requires domain knowledge of valid ranges per ...


In [8]:
from profiling_utils import surface_insights

surface_report = surface_insights(df)

print("Surface-level insights:")
for narrative in surface_report["narrative"]:
    print(f"- {narrative}")

with open(eda_summary_path, "a", encoding="utf-8") as f:
    f.write("\n\n" + "=" * 60 + "\n")
    f.write("SURFACE-LEVEL FEATURE INSIGHTS\n")
    f.write("=" * 60 + "\n")

    f.write("\n" + "-" * 60 + "\n")
    f.write("CATEGORICAL SUMMARY\n")
    f.write("-" * 60 + "\n")
    f.write(surface_report["categorical"].to_string(index=False))
    f.write("\n")

    f.write("\n" + "-" * 60 + "\n")
    f.write("NUMERIC SUMMARY\n")
    f.write("-" * 60 + "\n")
    f.write(surface_report["numeric"].to_string(index=False))
    f.write("\n")

    f.write("\n" + "-" * 60 + "\n")
    f.write("NARRATIVE INSIGHTS\n")
    f.write("-" * 60 + "\n")
    for narrative in surface_report["narrative"]:
        f.write(f"- {narrative}\n")

Surface-level insights:
- loan_id: 0.4% of non-null values are 'LP001032' -> low dominance; consider 'Unknown' category
- gender: 81.1% of non-null values are 'Male' -> mode imputation reasonable
- married: 64.0% of non-null values are 'Yes' -> mode imputation reasonable
- dependents: 57.8% of non-null values are '0' -> low dominance; consider 'Unknown' category
- education: 78.7% of non-null values are 'Graduate' -> mode imputation reasonable
- self_employed: 87.2% of non-null values are 'No' -> mode imputation reasonable
- property_area: 39.1% of non-null values are 'Semiurban' -> low dominance; consider 'Unknown' category
- applicant_income: mean=5379.37, median=3762.00, zero values=0.0% -> median imputation preferred (skewed)
- coapplicant_income: mean=1692.60, median=1250.00, zero values=43.3% -> median imputation preferred (skewed); zero-heavy; verify structural vs. missing
- loan_amount: mean=147.43, median=128.00, zero values=0.0% -> median imputation preferred (skewed)
- loan_

### Data Insights

**Preprocessing** data insights:  
  
- `dependents`: supports the binary feature `has_dependents` and a fill '0' or `has_dependents` = false  
- `education`: supports imputation of 'Graduate' as previously considered  
- `self_employed`: supports previous imputation 'No'  
- `property_area`: assuming the loans are for homes, this along with the correlation matrix support a low importance on this field  
- `applicant_income`, `coapplicant_income`, `loan_amount`: all being skewed is not surprising and supports `debt_income_ratio` feature to normalize the value in loan determination (preprocessing).  
    - income amounts averaging between 3-7k imply 'monthly_income'  
    - `loan_amount`s avg between 130-150, implying either the monthly payment amount or thousand(K) representation.  
    - `loan_amount_term`: appears to be either months or a 10x value of months, 36-48 months, the standard of loan terms. Supports the normalization ratio value to derive 'loan risk'.  

All other 'numerical' columns seem to follow previous assumptions for categories and binaries.


In [9]:
# Append dataset field charactistics to eda_summary.txt 
# in REPORTS_PATH

from profiling_utils import (
    generate_field_characteristics_report,
    write_missing_value_tables,
)

field_report = generate_field_characteristics_report(df)

missing_table_paths = write_missing_value_tables(
    field_report["missing_value_tables"],
    config.REPORTS_PATH,
)

with open(eda_summary_path, "a", encoding="utf-8") as f:
    f.write("\n\n" + "=" * 60 + "\n")
    f.write("ADDITIONAL FIELD CHARACTERISTICS\n")
    f.write("=" * 60 + "\n")

    for section_name, section_output in field_report.items():
        if isinstance(section_output, dict):
            for table_name, table in section_output.items():
                f.write("\n" + "-" * 60 + "\n")
                f.write(f"{table_name.upper()}\n")
                f.write("-" * 60 + "\n")

                if table.empty:
                    f.write("No rows identified.\n")
                else:
                    f.write(table.to_string(index=False))
                    f.write("\n")
        else:
            f.write("\n" + "-" * 60 + "\n")
            f.write(f"{section_name.upper()}\n")
            f.write("-" * 60 + "\n")

            if section_output.empty:
                f.write("No fields identified.\n")
            else:
                f.write(section_output.to_string(index=False))
                f.write("\n")

    if missing_table_paths:
        f.write("\n" + "-" * 60 + "\n")
        f.write("EXPORTED MISSING-VALUE TABLES\n")
        f.write("-" * 60 + "\n")
        for path in missing_table_paths:
            f.write(f"{path}\n")

print("field_characteristics_appended")

field_characteristics_appended


In [10]:
# Generate visualizations to FIGURE_PATH output folder

from profiling_utils import route_visualizations, render_visualizations
from utils import create_output_folders

create_output_folders(config.FIGURES_PATH)

visualization_plan = route_visualizations(
    df,
    config.TARGET_COLUMN,
    config.FIGURES_PATH,
)
saved_visualizations = render_visualizations(df, visualization_plan)

print(f"Generated {len(saved_visualizations)} visualizations:")
for path in saved_visualizations:
    print(f"- {path}")

with open(eda_summary_path, "a", encoding="utf-8") as f:
    f.write("\n\n" + "=" * 60 + "\n")
    f.write("GENERATED VISUALIZATIONS\n")
    f.write("=" * 60 + "\n")
    for item in visualization_plan:
        columns = ", ".join(map(str, item["columns"]))
        f.write(
            f"{item['type']}: {columns} -> {item['output_path']}\n"
        )

Generated 18 visualizations:
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\hist_applicant_income.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\box_applicant_income.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\hist_coapplicant_income.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\box_coapplicant_income.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\hist_loan_amount.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\box_loan_amount.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\hist_loan_amount_term.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\box_loan_amount_term.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\hist_credit_history.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\box_credit_history.png
- Q:\scripts\projects\Mock_Interview-July2026\outputs\figures\count_gender.png
- Q:\scripts\projects\Mock_